# Performance Leaderboard

This page provides an interactive view of the latest experimental results across different datasets and models. 

Results are aggregated from multiple runs and compared against the **OPLS-DA** baseline using a paired t-test.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from pathlib import Path
import os

# Fix for ReadTheDocs rendering
pio.renderers.default = 'notebook_connected'

# Load data generated by docs/generate_leaderboard_docs.py
csv_path = Path('_static') / 'leaderboard_data.csv'
if not csv_path.exists():
    csv_path = Path('.') / '_static' / 'leaderboard_data.csv'

if csv_path.exists():
    df = pd.read_csv(csv_path)
    # Ensure numeric columns are clean
    for col in ['Train', 'Test', 'Test Std', 'Train Std']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)
else:
    df = pd.DataFrame(columns=['Dataset', 'Method', 'Train', 'Test'])

def get_color_map(methods):
    colors = px.colors.qualitative.Plotly + px.colors.qualitative.Bold
    unique_methods = sorted(list(set(methods)))
    return {m: colors[i % len(colors)] for i, m in enumerate(unique_methods)}

if not df.empty:
    color_map = get_color_map(df['Method'].unique())
else:
    color_map = {}

## Summary Table

| Symbol | Meaning |
|---|---|
| `+` | Significantly better than OPLS-DA |
| `-` | Significantly worse than OPLS-DA |
| `≈` | No significant difference |

In [ ]:
if not df.empty:
    preferred_cols = ['Dataset', 'Method', 'Train', 'Test', 'Sig Te', 'Baseline']
    actual_cols = [c for c in preferred_cols if c in df.columns]
    pdf = df[actual_cols].sort_values(['Dataset', 'Test'], ascending=[True, False])
    
    # Create a Plotly Table for consistent dark/light mode rendering
    fig = go.Figure(data=[go.Table(
        header=dict(
            values=[f"<b>{c}</b>" for c in pdf.columns],
            fill_color='paleturquoise',
            align='left',
            font=dict(color='black', size=12)
        ),
        cells=dict(
            values=[pdf[c] for c in pdf.columns],
            format=[None, None, '.4f', '.4f', None, None],
            fill_color='lavender',
            align='left',
            font=dict(color='black', size=11)
        ))
    ])
    fig.update_layout(title="Statistical Significance Summary", margin=dict(l=0, r=0, t=30, b=0))
    fig.show()
else:
    print('No data available to display.')

## Comparative Performance
The bar charts below show the mean balanced accuracy for each method, with error bars representing one standard deviation across runs.

In [ ]:
if not df.empty:
    for ds in df['Dataset'].unique():
        ds_df = df[df['Dataset'] == ds].sort_values('Test', ascending=False)
        fig = px.bar(
            ds_df, x='Method', y='Test', error_y='Test Std' if 'Test Std' in ds_df.columns else None,
            title=f"Leaderboard: {ds.upper()}", 
            color='Method', 
            color_discrete_map=color_map,
            template='plotly_white',
            labels={'Test': 'Mean Balanced Accuracy'}
        )
        fig.update_layout(yaxis_range=[0, 1.05])
        fig.show()
else:
    print('No results available.')

## Training vs Testing Stability
This visualization highlights the gap between training and testing performance. A large gap may indicate over-fitting to the spectral signatures of the training set.

In [ ]:
if not df.empty:
    for ds in df['Dataset'].unique():
        ds_df = df[df['Dataset'] == ds].sort_values('Test', ascending=False)
        fig = go.Figure()
        fig.add_trace(go.Bar(
            x=ds_df['Method'], y=ds_df['Train'], name='Train',
            marker_color='rgba(100, 100, 100, 0.4)'
        ))
        fig.add_trace(go.Bar(
            x=ds_df['Method'], y=ds_df['Test'], name='Test',
            marker_color=[color_map.get(m, 'blue') for m in ds_df['Method']]
        ))
        fig.update_layout(
            barmode='overlay', 
            title=f"Train/Test Gap: {ds.upper()}",
            template='plotly_white', 
            yaxis_title='Balanced Accuracy',
            yaxis_range=[0, 1.05]
        )
        fig.show()
else:
    print('No results available.')